# Predicting cyclist traffic in Paris - EDA

Glenn Louis Opitz, Alexandre Violleau

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns

#sns.set_theme()

In [ ]:
data = pd.read_parquet(Path("data") / "train.parquet")
data.head()

One can observe that the `log_bike_count` column is calculated when applying $ln( \text{bike-count} + 1)$ to the `bike_count` column. The `+1` ensures that the function is still defined for `0`bikes. The idea of taking the natural log is to: 

1. Handle Skewness: Traffic data is ofton skewed with large number of hours with low counts and few hours with high counts. The log reduces the skewness and makes the data more normally distributed.

2. Stabilize variance

3. Better Model Interpretability: Log-transformed predictions allow for proportional interpretations. For example, a small cange in `log_bike_count` corresponds to a percentage change in the actual bike count.

In [ ]:
data.info()

In [ ]:
data.nunique(axis=0)

There are 30 counting sites with sometimes multiple counters per location. 

In [ ]:
(
    data.groupby(["site_name", "counter_name"])["bike_count"].sum()
    .sort_values(ascending=False)
    .head(10)
    .to_frame()
)

`Totem 73 boulevard de Sébastopol` appears twice on first and third place in terms of bike_counts.

## Visualizing the data 

In [ ]:
import folium

m = folium.Map(location=data[["latitude", "longitude"]].mean(axis=0), zoom_start=13)

for _, row in (
    data[["counter_name", "latitude", "longitude"]]
    .drop_duplicates("counter_name")
    .iterrows()
):
    folium.Marker(
        row[["latitude", "longitude"]].values.tolist(), popup=row["counter_name"]
    ).add_to(m)

m

In [ ]:
mask = data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N"

# Aggregate the data
data_mask = data[mask].copy()
data_mask["date"] = pd.to_datetime(data_mask["date"])  # Ensure date is in datetime format
data_agg = data_mask.groupby("date", as_index=False)["bike_count"].sum()

data_agg.plot(x="date", y="bike_count", title="Bike Count Over Time", legend=True)


In [ ]:
mask = (data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")

data[mask].groupby(
    pd.Grouper(freq="1w", key="date")
)[["bike_count"]].sum().plot()

Zooming in one week:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Filter for the specific counter and date range
mask = (
    (data["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (data["date"] > pd.to_datetime("2021-03-01"))
    & (data["date"] < pd.to_datetime("2021-03-08"))
)

# Aggregate data to daily totals (if needed)
data_filtered = data[mask].copy()
data_filtered = data_filtered.groupby("date", as_index=False)["bike_count"].sum()

# Plot the data
data_filtered.plot(x="date", y="bike_count", ax=ax, marker='.', legend=False)
ax.set_title("Bike Count from March 1 to March 8, 2021")
ax.set_ylabel("Bike Count")
ax.set_xlabel("Date")
plt.show()



There appears to be an hourly pattern during workdays with several peaks a day. On weekends, like on 7th and 8th March, the shape of the during the day looks different. In terms of daily peaks, there seem to be slight differences when comparing Monday-Wednesday with Thursday & Friday.

So let us now take a look at all bike counts aggregated on each weekday.

In [ ]:
def _encode_dates(X):
    X = X.copy()  # Ensure we're working on a copy
    # Encode the date information
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour
    # Keep the rest of the columns as they are
    return X

# Apply the encoding function to the dataset
data = data.copy()  # Ensure we're working on a copy
data = _encode_dates(data)


In [ ]:
# Aggregate bike counts by weekday using the encoded 'weekday' column
weekday_aggregates = data.groupby("weekday")["bike_count"].sum()

# Reorder the weekdays to start from Monday (0 = Monday, ..., 6 = Sunday)
weekday_order = [0, 1, 2, 3, 4, 5, 6]
weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_aggregates = weekday_aggregates.reindex(weekday_order)

# Plot as a bar chart
plt.figure(figsize=(10, 6))
weekday_aggregates.index = weekday_names  # Replace index with weekday names
weekday_aggregates.plot(kind="bar", color="lightblue", edgecolor="black")
plt.title("Total Bike Count by Weekday")
plt.ylabel("Total Bike Count")
plt.xlabel("Day of the Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

We can see differences among the weekdays on an aggregate level (all counters and along the whole time frame). It would be interesting to see, whether this distribution holds for each counter individually.

In [ ]:
# Group by station and weekday, summing bike counts
station_weekday_counts = data.groupby(["counter_name", "weekday"])["bike_count"].sum().unstack()

# Normalize counts by Monday (weekday 0)
normalized_counts = station_weekday_counts.div(station_weekday_counts[0], axis=0) * 100

# Plot the normalized distributions for all stations
plt.figure(figsize=(12, 6))
for station in normalized_counts.index:
    plt.plot(normalized_counts.columns, normalized_counts.loc[station], label=station, alpha=0.6)

# Add labels and legend
plt.title("Normalized Bike Counts by Weekday (Percentage of Monday) for each counter")
plt.ylabel("Percentage of Monday's Count (%)")
plt.xlabel("Weekday (0=Monday, ..., 6=Sunday)")
#plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
plt.tight_layout()
plt.show();

It appears that the rough distribution over weekdays, examined before, holds for all counters. In terms of percentage change against monday (indexed), the weekdays seem most consistent. On weekend days, the variation amongst counters is much higher and may need better investigation.

Maybe it makes sense to group Tuesday until Thursday as one category, given they are all at an equal level in terms of counts.

For least square loss, normal error distributions are beneficial.

In [ ]:
ax = sns.histplot(data, x="bike_count", kde=True, bins=50)

Due to the skewed distribution of `bike_count`, using `log_bike_count` already improves the shape of the distribution, while still not being perfect.

In [ ]:
ax = sns.histplot(data, x="log_bike_count", kde=True, bins=50)

## Correlation analysis

In [ ]:
# Select relevant features for correlation analysis
correlation_features = ["log_bike_count", "hour", "weekday", "month", "year", "bike_count"]

# Compute the correlation matrix
correlation_matrix = data[correlation_features].corr()

# Plot the correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap with Log Bike Count")
plt.show()

For `log_bike_count`, the strongest correlation can be observed with `hour`, suggesting a deeper analysis on an hourly basis. Presumably, it will make sense to create categorical variables that encode phases of the day (e.g. morning, lunch, afternoon, after-work,...).

In [ ]:
# Define a function to create time-of-day phases
def _add_time_phases(X):
    X = X.copy()  # Work on a copy
    # Create a 'time_of_day' categorical variable
    X["time_of_day"] = pd.cut(
        X["hour"],
        bins=[0, 5, 9, 13, 17, 20, 23],  # Fix the upper boundary to match the bins
        labels=["Night", "Morning", "Midday", "Afternoon", "Evening", "Late Evening"],  # Corresponding labels
        right=True
    )
    return X

# Apply the function to the dataset
data = _add_time_phases(data)

# Preview the new column
print(data[["hour", "time_of_day"]].head())


In [ ]:
# Group by 'time_of_day' and calculate mean and median of 'log_bike_count'
time_of_day_stats = data.groupby("time_of_day")["log_bike_count"].agg(["mean", "median"]).reset_index()

# Group by 'hour' and calculate mean 'log_bike_count'
hourly_stats = data.groupby("hour")["log_bike_count"].mean().reset_index()

# Separate the hourly mean and time-of-day mean plots for clarity
fig, ax = plt.subplots(2, 1, figsize=(8, 8), sharey=True)

# Plot hourly mean
ax[0].plot(hourly_stats["hour"], hourly_stats["log_bike_count"], label="Hourly Mean", marker="o")
ax[0].set_title("Hourly Mean of Log Bike Count")
ax[0].set_ylabel("Log Bike Count")
ax[0].set_xlabel("Hour of the Day")
ax[0].grid(True)

# Plot time-of-day mean
time_of_day_stats.plot(
    x="time_of_day",
    y="mean",
    kind="bar",
    ax=ax[1],
    color="red",
    legend=False,
    edgecolor="black",
)
ax[1].set_title("Time of Day Mean of Log Bike Count")
ax[1].set_ylabel("Log Bike Count")
ax[1].set_xlabel("Time of Day")
ax[1].set_xticks(range(len(time_of_day_stats["time_of_day"])))
ax[1].set_xticklabels(time_of_day_stats["time_of_day"], rotation=45)

plt.tight_layout()
plt.show()


Observing the hourly mean vs. the mean per day phase shows, that the phases cannot really capture hourly differences. Thus, they might simplify to much and leave out information.

## Adding external weather data

In [ ]:
weather_data = pd.read_csv(Path("data") / "external_data.csv")
print(weather_data.info())
print(weather_data.head())


In [ ]:
weather_data["date"] = pd.to_datetime(weather_data["date"], errors="coerce")

print(weather_data["date"].isna().sum())

weather_data = _encode_dates(weather_data)



In [ ]:
# Merge the datasets
merged_data = pd.merge(data, weather_data, on=["year", "month", "day", "hour"], how="inner")

print(merged_data.head())

In [ ]:
# Check the shape of the merged dataset
print(f"Merged Data Shape: {merged_data.shape}")
print(f"Original Bike Data Shape: {data.shape}")
print(f"Original Weather Data Shape: {weather_data.shape}")


No proper merge because Bike Data has several rows for the same point in time. For this reason, for EDA, we will aggregate per day:

In [ ]:
aggregated_bike_data = data.groupby("date").agg({
    "bike_count": "sum"  # Sum total bike counts for all counters
}).reset_index()

# Compute the logarithm of the aggregated bike counts
aggregated_bike_data["log_bike_count_aggregated"] = np.log(aggregated_bike_data["bike_count"] + 1)

# Check the resulting aggregated data
print(aggregated_bike_data.head())


In [ ]:
# Merge weather_data with aggregated bike data
merged_data = pd.merge(aggregated_bike_data, weather_data, on="date", how="inner")

print(f"Merged Data Shape: {merged_data.shape}")
print(f"Aggregated Bike Data Shape: {aggregated_bike_data.shape}")
print(f"Weather Data Shape: {weather_data.shape}")


In [ ]:
# Unique timestamps in each dataset
bike_timestamps = set(aggregated_bike_data["date"])
weather_timestamps = set(weather_data["date"])

# Check overlaps
overlap = len(bike_timestamps & weather_timestamps)
print(f"Number of overlapping timestamps: {overlap}")
print(f"Unique timestamps in bike data: {len(bike_timestamps)}")
print(f"Unique timestamps in weather data: {len(weather_timestamps)}")


In [ ]:
# Missing in weather data
missing_in_weather = bike_timestamps - weather_timestamps
print(f"Timestamps in bike data missing from weather data: {len(missing_in_weather)}")

# Missing in bike data
missing_in_bike = weather_timestamps - bike_timestamps
print(f"Timestamps in weather data missing from bike data: {len(missing_in_bike)}")


In [ ]:
# Check the frequency of weather data timestamps
print(weather_data["date"].diff().value_counts())


The bike data is hourly while the weather data is every 3 hours. To not loose information in our bike data set, we will resample the weather data to hourly by filling the gaps in the weather data using interpolation.

In [ ]:
# Check for duplicate timestamps in the weather data
duplicate_dates = weather_data["date"].duplicated().sum()
print(f"Number of duplicate timestamps in weather data: {duplicate_dates}")


In [ ]:
# Get the duplicate timestamps
duplicate_rows = weather_data[weather_data["date"].duplicated(keep=False)]
print(duplicate_rows)


In [ ]:
# Drop the duplicate row based on the 'date' column
weather_data = weather_data.drop_duplicates(subset="date")

# Verify that the duplicate is removed
duplicate_rows = weather_data[weather_data["date"].duplicated(keep=False)]
print(f"Number of duplicate rows after dropping: {len(duplicate_rows)}")


In [ ]:
weather_data.set_index("date", inplace=True)  # Set date as the index
weather_data = weather_data.resample("H").interpolate(method="linear")  # Interpolate missing values
weather_data.reset_index(inplace=True)  # Reset index

# Check the new shape of the weather data
print(f"Resampled Weather Data Shape: {weather_data.shape}")
print(weather_data.head())


In [ ]:
# Verify the frequency of timestamps after resampling
print(weather_data["date"].diff().value_counts())


In [ ]:
# Update weather timestamps after resampling
weather_timestamps = set(weather_data["date"])

# Recalculate overlap with bike timestamps
overlap = len(bike_timestamps & weather_timestamps)
print(f"Number of overlapping timestamps after resampling: {overlap}")


In [ ]:
# Merge the aggregated bike data with the resampled weather data
merged_data = pd.merge(aggregated_bike_data, weather_data, on="date", how="inner")

# Check the shape and preview the merged data
print(f"Merged Data Shape: {merged_data.shape}")
print(merged_data.head())


In [ ]:
# Filter correlations for 'log_bike_count_aggregated' greater than the threshold (|0.1|)
threshold = 0.1
correlations = correlation_matrix["log_bike_count_aggregated"].abs()
filtered_variables = correlations[correlations > threshold].index

# Create a filtered correlation matrix
filtered_correlation_matrix = correlation_matrix.loc[filtered_variables, filtered_variables]

# Plot the filtered heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(filtered_correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Filtered Correlation Heatmap (Threshold: |0.2|)")
plt.show()


In [ ]:
# Select top N variables by correlation strength
top_variables = correlations.sort_values(ascending=False).index[1:11]  # Top 10 excluding the target

# Include 'log_bike_count_aggregated' in the pairwise plot
top_variables_with_target = ["log_bike_count_aggregated"] + top_variables.tolist()

# Create pairwise scatterplots including the target variable
sns.pairplot(merged_data, vars=top_variables_with_target, diag_kind="kde", kind="scatter", corner=True)
plt.suptitle("Pairwise Scatterplots Including log_bike_count_aggregated", y=1.02)
plt.show()



In [ ]:
correlations.sort_values(ascending=False)

The most correlated columns (with correlation coefficient of > |0.1| are:

| Variable  | Description                           | Unit          |
|-----------|---------------------------------------|---------------|
| u         | Humidity                              | %             |
| t         | Temperature                           | K             |
| tx12      | Max. temperature over 12 hours        | K             |
| tn12      | Min. temperature over 12 hours        | K             |
| Rafper    | Bursts over a period                  | m/s           |
| td        | Dew point                             | K             |
| raf10     | Burst over last 10 minutes            | m/s           |
| ff        | Average wind speed over 10 minutes    | m/s           |
| nnuage3   | Cloud cover layer 3                   | octa          |
| nnuage2   | Cloud cover layer 2                   | octa          |
| nnuage1   | Cloud cover layer 1                   | octa          |
| ww        | Present weather condition             | N/A           |
| etat_sol  | Soil condition                        | N/A           |
| hnuage4   | Cloud base height for layer 4         | m             |
| vv        | Horizontal visibility                 | m             |